# SABIO-RK Reaction 618 beta-glucosidase pilot

This notebook uses the local SABIO-RK Reaction 618 fixture and the registry-backed FungMod screening API. It keeps the literature-curated scientific case underparameterized when enzyme concentration is missing, then runs a clearly marked exploratory homogeneous Michaelis-Menten ensemble.

This exploratory ensemble uses a user-supplied enzyme-concentration range.
The enzyme concentration is not curated from SABIO-RK EntryID 35622.
This remains an enzyme-only kinetic pilot, not a whole-fungus degradation model.

In [ ]:
from pathlib import Path
import csv
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data_registry" / "registry_index.yml").exists():
    ROOT = Path("..").resolve()

src_path = ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from fungal_model.data import load_kinetic_record
from fungal_model.registry import load_registry
from fungal_model.screening import assess_modelability, simulate_screen

In [ ]:
FUNGUS_ID = "sabiork_beta_glucosidase_source"
SUBSTRATE_ID = "cellobiose"
ENVIRONMENT_ID = "sabiork_reaction_618_selected_conditions"
ENZYME_CONCENTRATION_SYMBOL = "enzyme_concentration_beta_glucosidase"

REGISTRY_INDEX = ROOT / "data_registry" / "registry_index.yml"
KINETIC_RECORD_PATH = (
    ROOT
    / "data"
    / "kinetic_records"
    / "sabiork"
    / "case_001_reaction_618_beta_glucosidase"
    / "curated"
    / "kinetic_record.yml"
)

In [ ]:
registry = load_registry(REGISTRY_INDEX)
record = load_kinetic_record(KINETIC_RECORD_PATH)

{
    "source_database": record.source_database,
    "reaction_id": record.source_reaction_id,
    "selected_entry_id": record.source_kinetic_law_id,
    "enzyme": record.enzyme.name,
    "reaction": record.reaction.equation,
}

In [ ]:
report = assess_modelability(
    fungus_id=FUNGUS_ID,
    substrate_id=SUBSTRATE_ID,
    environment_id=ENVIRONMENT_ID,
    registry=registry,
    mode="scientific",
)

report.to_dict()

In [ ]:
exploratory_prior = next(
    parameter
    for parameter in registry.get_parameter_records(
        parameter_symbol=ENZYME_CONCENTRATION_SYMBOL,
        process_type="homogeneous_michaelis_menten",
    )
    if parameter.maturity == "exploratory_prior"
)

{
    "record_id": exploratory_prior.record_id,
    "maturity": exploratory_prior.maturity,
    "value": exploratory_prior.value.to_dict(),
    "exploratory_prior": exploratory_prior.provenance.get("exploratory_prior"),
}

In [ ]:
N_SAMPLES = 32
SEED = 1
OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", str(ROOT / "outputs")))
screen_output_dir = OUTPUT_ROOT / "sabiork_reaction_618_homogeneous_mm_ensemble"

screen = simulate_screen(
    fungus_ids=[FUNGUS_ID],
    substrate_ids=[SUBSTRATE_ID],
    environment_ids=[ENVIRONMENT_ID],
    registry=registry,
    mode="exploratory",
    n_samples=N_SAMPLES,
    seed=SEED,
    output_dir=screen_output_dir,
)

screen.to_dict()["case_results"][0]["modelability_report"]["status"]

In [ ]:
def read_csv_rows(path, *, limit=None):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    return rows if limit is None else rows[:limit]


sampled_parameter_rows = read_csv_rows(screen_output_dir / "sampled_parameters.csv")
sampled_parameter_rows[:5]

In [ ]:
final_state_rows = read_csv_rows(screen_output_dir / "final_states.csv")
final_state_rows[:5]

In [ ]:
import matplotlib.pyplot as plt

enzyme_values = [
    float(row[ENZYME_CONCENTRATION_SYMBOL])
    for row in sampled_parameter_rows
]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(enzyme_values, bins=12)
ax.set_xscale("log")
ax.set_xlabel("enzyme_concentration_beta_glucosidase (mM)")
ax.set_ylabel("sample count")
ax.set_title("Sampled exploratory enzyme concentration")
fig.tight_layout()
fig

In [ ]:
final_glucose_values = [
    float(row["final_beta_D_glucose_concentration"])
    for row in final_state_rows
]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(final_glucose_values, bins=12)
ax.set_xlabel("final_beta_D_glucose_concentration (mM)")
ax.set_ylabel("sample count")
ax.set_title("Final beta-D-glucose concentration")
fig.tight_layout()
fig

In [ ]:
final_cellobiose_values = [
    float(row["final_cellobiose_concentration"])
    for row in final_state_rows
]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(final_cellobiose_values, bins=12)
ax.set_xlabel("final_cellobiose_concentration (mM)")
ax.set_ylabel("sample count")
ax.set_title("Final cellobiose concentration")
fig.tight_layout()
fig

## Limitations

The SABIO-RK selected kinetic-law entry does not provide enzyme concentration, so scientific modelability remains underparameterized. The ensemble range is a user-supplied exploratory prior, not a literature-curated SABIO-RK value. This notebook does not model fungus growth, secretion, uptake, biomass, oxygen limitation, PET chemistry, cellulose surface morphology, or time-course validation data.